In [ ]:
import pandas as pd
from pathlib import Path
import sys

sys.path.append("../src")

from usage_processor import UsageProcessor

processor = UsageProcessor(
    "../../data/sms-call-internet-mi-2013-11-01.csv"
)

In [11]:
# 1. Start from the grid/hour analytics table produced by aggregate_to_grid_time() in NP2.

from pathlib import Path
import sys

sys.path.append("../src")

from usage_processor import UsageProcessor


processor = UsageProcessor(
    "../../data/sms-call-internet-mi-2013-11-01.csv"
)

processor.load_data()
processor.clean_data()
processor.derive_time_features()
processor.aggregate_to_grid_time()

grid_hour_df = processor.df

print("Shape:", grid_hour_df.shape)

print("\nColumns:")
print(grid_hour_df.columns.tolist())

print("\nFirst 5 rows:")
print(grid_hour_df.head())

Shape: (240000, 10)

Columns:
['datetime', 'date', 'hour', 'day_of_week', 'CellID', 'smsin', 'smsout', 'callin', 'callout', 'internet']

First 5 rows:
    datetime        date  hour day_of_week  CellID   smsin  smsout  callin  \
0 2013-11-01  2013-11-01     0      Friday       1  2.0843  1.1047  0.5919   
1 2013-11-01  2013-11-01     0      Friday       2  2.0915  1.0880  0.6020   
2 2013-11-01  2013-11-01     0      Friday       3  2.0992  1.0701  0.6128   
3 2013-11-01  2013-11-01     0      Friday       4  2.0633  1.1533  0.5627   
4 2013-11-01  2013-11-01     0      Friday       5  1.8708  1.0439  0.5110   

   callout  internet  
0   0.4293   57.7990  
1   0.4382   57.9149  
2   0.4476   58.0382  
3   0.4036   57.4634  
4   0.3740   52.1714  


In [13]:
# 2. Build the within-day baseline: for each grid_id, compute the median total_activity across that day’s 24 hourly intervals, excluding the hour currently being evaluated. Use the median rather than the mean so a single extreme hour does not raise its own baseline.

processor.derive_activity_features()

grid_hour_df = processor.df

print(grid_hour_df[
    ["datetime", "CellID", "total_activity"]
].head())

    datetime  CellID  total_activity
0 2013-11-01       1         62.0092
1 2013-11-01       2         62.1346
2 2013-11-01       3         62.2679
3 2013-11-01       4         61.6463
4 2013-11-01       5         55.9711


In [14]:
# 2. Build the within-day baseline: for each grid_id, compute the median total_activity across that day’s 24 hourly intervals, excluding the hour currently being evaluated. Use the median rather than the mean so a single extreme hour does not raise its own baseline.

baseline_values = []

for index, row in grid_hour_df.iterrows():

    grid_id = row["CellID"]

    other_hours = grid_hour_df[
        (grid_hour_df["CellID"] == grid_id) &
        (grid_hour_df.index != index)
    ]["total_activity"]

    baseline_values.append(other_hours.median())

grid_hour_df["baseline_activity"] = baseline_values

In [15]:
print(
    grid_hour_df[
        ["datetime", "CellID", "total_activity", "baseline_activity"]
    ].head(10)
)

    datetime  CellID  total_activity  baseline_activity
0 2013-11-01       1         62.0092            83.7437
1 2013-11-01       2         62.1346            84.2422
2 2013-11-01       3         62.2679            84.7727
3 2013-11-01       4         61.6463            82.2863
4 2013-11-01       5         55.9711            75.2243
5 2013-11-01       6         62.2679            84.7727
6 2013-11-01       7         62.2679            84.7727
7 2013-11-01       8         62.2679            84.7727
8 2013-11-01       9         62.2679            84.7727
9 2013-11-01      10         36.0330            52.4707


In [16]:
grid_1 = grid_hour_df[
    grid_hour_df["CellID"] == 1
].sort_values("datetime")

print(grid_1[
    ["datetime", "total_activity", "baseline_activity"]
])

                  datetime  total_activity  baseline_activity
0      2013-11-01 00:00:00         62.0092            83.7437
10000  2013-11-01 01:00:00         46.3654            83.7437
20000  2013-11-01 02:00:00         42.0870            83.7437
30000  2013-11-01 03:00:00         35.0978            83.7437
40000  2013-11-01 04:00:00         32.2741            83.7437
50000  2013-11-01 05:00:00         35.4160            83.7437
60000  2013-11-01 06:00:00         36.1216            83.7437
70000  2013-11-01 07:00:00         44.7950            83.7437
80000  2013-11-01 08:00:00         67.4200            83.7437
90000  2013-11-01 09:00:00         83.7437            70.8246
100000 2013-11-01 10:00:00        100.7637            70.8246
110000 2013-11-01 11:00:00         95.1478            70.8246
120000 2013-11-01 12:00:00         91.2744            70.8246
130000 2013-11-01 13:00:00         92.0887            70.8246
140000 2013-11-01 14:00:00        103.0604            70.8246
150000 2

In [17]:
# 3. Apply an activity floor. Grids with very low daily totals produce meaningless ratios and will otherwise dominate the alert list. Choose the floor from the data and document the choice.

daily_grid_activity = (
    grid_hour_df
    .groupby("CellID")["total_activity"]
    .sum()
    .sort_values()
)

print("=== DAILY ACTIVITY DISTRIBUTION ===")

print(daily_grid_activity.describe())

print("\nLowest 20 grids:")
print(daily_grid_activity.head(20))

print("\nHighest 20 grids:")
print(daily_grid_activity.tail(20))

=== DAILY ACTIVITY DISTRIBUTION ===
count     10000.000000
mean       9572.762612
std       14153.589167
min          55.179000
25%        1953.600725
50%        4979.452150
75%       10510.519875
max      274800.295600
Name: total_activity, dtype: float64

Lowest 20 grids:
CellID
2801     55.1790
112      69.7055
1207     97.3609
5310    129.1392
5210    129.1749
4906    129.3575
5007    134.4480
5209    139.6356
5309    145.1330
5409    150.1235
5410    163.3828
5308    163.8013
4905    165.6071
5109    169.5613
5108    173.9599
5508    175.7698
5006    177.5808
5208    185.1050
5408    188.6380
5008    189.0190
Name: total_activity, dtype: float64

Highest 20 grids:
CellID
5567    105790.0871
5857    106148.4458
5855    106165.8445
6072    106965.5927
5658    107140.1977
5956    109792.7562
6073    110620.4230
4654    111520.1209
4755    112272.0130
5458    113380.8117
4856    114005.3370
4857    121991.8232
5162    124664.3217
6064    126818.9053
5262    131322.8759
5955    135844.

In [18]:
# 3. Apply an activity floor. Grids with very low daily totals produce meaningless ratios and will otherwise dominate the alert list. Choose the floor from the data and document the choice.

ACTIVITY_FLOOR = daily_grid_activity.quantile(0.25)

print("Activity floor:", ACTIVITY_FLOOR)

grid_hour_df["daily_total_activity"] = (
    grid_hour_df
    .groupby("CellID")["total_activity"]
    .transform("sum")
)

grid_hour_df["above_activity_floor"] = (
    grid_hour_df["daily_total_activity"] >= ACTIVITY_FLOOR
)

print(
    "Grids above activity floor:",
    grid_hour_df.loc[
        grid_hour_df["above_activity_floor"], "CellID"
    ].nunique()
)

print(
    "Grids below activity floor:",
    grid_hour_df.loc[
        ~grid_hour_df["above_activity_floor"], "CellID"
    ].nunique()
)

Activity floor: 1953.600725
Grids above activity floor: 7500
Grids below activity floor: 2500


### Activity Floor Decision

The activity floor is set at the 25th percentile of daily grid activity,
which is 1953.600725.

Grids below this floor are considered very low-activity grids and are
excluded from rule-based alert generation. The grids remain in the
analytics dataset; the floor is applied only when evaluating alerts.

The floor was selected from the supplied day's activity distribution
rather than using an arbitrary fixed value.

In [19]:
# 4. Define three training rules against that baseline: HIGH_ACTIVITY (current hour materially above the grid’s own within-day baseline), ACTIVITY_SPIKE (sharp rise against the immediately preceding hour) and ACTIVITY_DROP (current hour materially below the grid’s own baseline).

grid_hour_df = grid_hour_df.sort_values(
    ["CellID", "datetime"]
).copy()

grid_hour_df["previous_activity"] = (
    grid_hour_df
    .groupby("CellID")["total_activity"]
    .shift(1)
)

In [20]:
HIGH_ACTIVITY_THRESHOLD = 1.50
SPIKE_THRESHOLD = 1.50
DROP_THRESHOLD = 0.50

In [21]:
grid_hour_df["high_activity"] = (
    grid_hour_df["total_activity"]
    >= grid_hour_df["baseline_activity"] * HIGH_ACTIVITY_THRESHOLD
)

grid_hour_df["activity_spike"] = (
    grid_hour_df["previous_activity"].notna()
    & (
        grid_hour_df["total_activity"]
        >= grid_hour_df["previous_activity"] * SPIKE_THRESHOLD
    )
)

grid_hour_df["activity_drop"] = (
    grid_hour_df["total_activity"]
    <= grid_hour_df["baseline_activity"] * DROP_THRESHOLD
)

In [22]:
print("HIGH_ACTIVITY alerts:", grid_hour_df["high_activity"].sum())
print("ACTIVITY_SPIKE alerts:", grid_hour_df["activity_spike"].sum())
print("ACTIVITY_DROP alerts:", grid_hour_df["activity_drop"].sum())

HIGH_ACTIVITY alerts: 15966
ACTIVITY_SPIKE alerts: 5830
ACTIVITY_DROP alerts: 25922


In [23]:
# 5. For each grid and hour, compare current activity against the baseline and apply the three rules.

alerts = []

for _, row in grid_hour_df.iterrows():

    # Skip very low-activity grids
    if not row["above_activity_floor"]:
        continue

    current_activity = row["total_activity"]
    baseline_activity = row["baseline_activity"]
    previous_activity = row["previous_activity"]

    # HIGH_ACTIVITY
    if row["high_activity"]:
        alerts.append({
            "CellID": row["CellID"],
            "datetime": row["datetime"],
            "alert_type": "HIGH_ACTIVITY",
            "current_activity": current_activity,
            "baseline_activity": baseline_activity,
            "reason": (
                f"HIGH_ACTIVITY: current activity {current_activity:.2f} "
                f"is at least 50% above baseline {baseline_activity:.2f}."
            )
        })

    # ACTIVITY_SPIKE
    if row["activity_spike"]:
        alerts.append({
            "CellID": row["CellID"],
            "datetime": row["datetime"],
            "alert_type": "ACTIVITY_SPIKE",
            "current_activity": current_activity,
            "baseline_activity": baseline_activity,
            "reason": (
                f"ACTIVITY_SPIKE: current activity {current_activity:.2f} "
                f"is at least 50% above previous-hour activity "
                f"{previous_activity:.2f}."
            )
        })

    # ACTIVITY_DROP
    if row["activity_drop"]:
        alerts.append({
            "CellID": row["CellID"],
            "datetime": row["datetime"],
            "alert_type": "ACTIVITY_DROP",
            "current_activity": current_activity,
            "baseline_activity": baseline_activity,
            "reason": (
                f"ACTIVITY_DROP: current activity {current_activity:.2f} "
                f"is at least 50% below baseline {baseline_activity:.2f}."
            )
        })

alerts_df = pd.DataFrame(alerts)

print("Total alert records:", len(alerts_df))
print("\nAlerts by type:")
print(alerts_df["alert_type"].value_counts())

print("\nFirst 10 alerts:")
print(alerts_df.head(10))

Total alert records: 33326

Alerts by type:
alert_type
ACTIVITY_DROP     18483
HIGH_ACTIVITY     10731
ACTIVITY_SPIKE     4112
Name: count, dtype: int64

First 10 alerts:
   CellID            datetime      alert_type  current_activity  \
0      65 2013-11-01 04:00:00   ACTIVITY_DROP           39.1525   
1      65 2013-11-01 05:00:00   ACTIVITY_DROP           35.7342   
2      65 2013-11-01 06:00:00   ACTIVITY_DROP           36.9632   
3      65 2013-11-01 07:00:00   ACTIVITY_DROP           46.6596   
4      65 2013-11-01 09:00:00  ACTIVITY_SPIKE           92.9208   
5      65 2013-11-01 18:00:00   HIGH_ACTIVITY          142.8166   
6      66 2013-11-01 04:00:00   ACTIVITY_DROP           39.1136   
7      66 2013-11-01 05:00:00   ACTIVITY_DROP           36.3657   
8      66 2013-11-01 06:00:00   ACTIVITY_DROP           37.4885   
9      66 2013-11-01 07:00:00   ACTIVITY_DROP           47.0342   

   baseline_activity                                             reason  
0            94.9

In [24]:
# 6. Create alert records containing grid_id, timestamp, alert_type, current_activity, baseline_activity and reason.

required_alert_columns = [
    "CellID",
    "datetime",
    "alert_type",
    "current_activity",
    "baseline_activity",
    "reason"
]

print("=== ALERT RECORD VALIDATION ===")

print("\nRequired columns present:")
for column in required_alert_columns:
    print(f"{column}: {column in alerts_df.columns}")

print("\nMissing values:")
print(alerts_df[required_alert_columns].isnull().sum())

print("\nAlert types:")
print(alerts_df["alert_type"].value_counts())

=== ALERT RECORD VALIDATION ===

Required columns present:
CellID: True
datetime: True
alert_type: True
current_activity: True
baseline_activity: True
reason: True

Missing values:
CellID               0
datetime             0
alert_type           0
current_activity     0
baseline_activity    0
reason               0
dtype: int64

Alert types:
alert_type
ACTIVITY_DROP     18483
HIGH_ACTIVITY     10731
ACTIVITY_SPIKE     4112
Name: count, dtype: int64


In [25]:
# 7. Write alerts to CSV or JSON and print a short operational summary:
# alerts by type, top ten grids by alert count, and the proportion of all grid/hours that alerted.

from pathlib import Path

# Create output directory
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save alerts
alert_file = output_dir / "network_alerts.csv"
alerts_df.to_csv(alert_file, index=False)

# Alerts by type
alerts_by_type = alerts_df["alert_type"].value_counts()

# Top 10 grids by alert count
top_10_grids = (
    alerts_df["CellID"]
    .value_counts()
    .head(10)
)

# Count unique grid/hour combinations that alerted
alerted_grid_hours = (
    alerts_df[["CellID", "datetime"]]
    .drop_duplicates()
    .shape[0]
)

# Total grid/hour combinations
total_grid_hours = len(grid_hour_df)

# Proportion of grid/hours that alerted
alert_proportion = (
    alerted_grid_hours / total_grid_hours
)

print("=== OPERATIONAL ALERT SUMMARY ===")

print("\nAlerts by type:")
print(alerts_by_type)

print("\nTop 10 grids by alert count:")
print(top_10_grids)

print("\nUnique grid/hours that alerted:")
print(alerted_grid_hours)

print("\nTotal grid/hours:")
print(total_grid_hours)

print(
    f"\nProportion of grid/hours that alerted: "
    f"{alert_proportion:.2%}"
)

print(f"\nAlert file written to: {alert_file}")

=== OPERATIONAL ALERT SUMMARY ===

Alerts by type:
alert_type
ACTIVITY_DROP     18483
HIGH_ACTIVITY     10731
ACTIVITY_SPIKE     4112
Name: count, dtype: int64

Top 10 grids by alert count:
CellID
7822    29
7823    29
7647    28
1583    27
1683    27
7724    27
1584    26
1684    26
3222    26
8169    26
Name: count, dtype: int64

Unique grid/hours that alerted:
31925

Total grid/hours:
240000

Proportion of grid/hours that alerted: 13.30%

Alert file written to: ..\outputs\network_alerts.csv
